# Computer lab 4: stochastic optimization

**Mathematical Optimization, VU Amsterdam, 2026–27.**

If time is limited, work through 1(a), 1(d), 1(f) and 1(g), then 3(a)–(c)
and Exercise 4. Return to the other parts of Exercise 1 and the newsvendor
in Exercise 2 afterward. Exercise 2(e) reproduces the SAA plot in L8.

Fill in the marked sections as you work. Remove the `NotImplementedError`
line after completing each function.

The examples draw on the
[MO-book companion notebooks](https://mobook.github.io/MO-book/intro.html)
([MIT license](https://github.com/mobook/MO-book/blob/ef77f6b9768453c300d5d97e42e89fce1f5e832e/LICENSE)).

## Setup

Run the next two cells to prepare Pyomo and HiGHS. If prompted to restart
the session, restart it and rerun both cells.
See the [course setup guide](https://github.com/alessandrozocca/MO2026#computer-lab-setup)
for help.

Use `solve_and_load(model, lp_solver)` for each solve. It checks for an
optimal result before loading the solution.

In [ ]:
#@title Prepare course helpers and solvers
import hashlib
import os
from pathlib import Path
import sys
import time
import types
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

SETUP_COMMIT = '425c3f0af23e082c276df85fcde556538cc2abef'
SETUP_SHA256 = "f742478c6eaf5c5db82f4ce69b47a2adcf6f920765d88c3e7eb1b1ab72e66d28"
local_repo = os.environ.get("MO2026_REPO")
if local_repo:
    setup_location = str(Path(local_repo).expanduser() / "lab_setup.py")
    setup_source = Path(setup_location).read_bytes()
else:
    setup_location = f"https://raw.githubusercontent.com/alessandrozocca/MO2026/{SETUP_COMMIT}/lab_setup.py"
    for attempt in range(3):
        try:
            with urlopen(setup_location, timeout=30) as response:
                setup_source = response.read()
            break
        except (URLError, TimeoutError, ConnectionError) as error:
            if (isinstance(error, HTTPError) and error.code not in (408, 429, 500, 502, 503, 504)) or attempt == 2:
                raise
            time.sleep(0.5 * (attempt + 1))
    if hashlib.sha256(setup_source).hexdigest() != SETUP_SHA256:
        raise RuntimeError("Course setup checksum mismatch; the downloaded code was not executed.")

lab_setup = types.ModuleType("mo2026_lab_setup")
lab_setup.__file__ = setup_location
sys.modules[lab_setup.__name__] = lab_setup
exec(compile(setup_source, setup_location, "exec"), lab_setup.__dict__)
if local_repo and lab_setup.API_VERSION != 1:
    raise RuntimeError("This notebook requires course setup API 1.")
lab = lab_setup.setup(week=4, check=False, verbose=True)
solve_and_load = lab_setup.solve_and_load
lp_solver = lab.lp_solver

In [ ]:
%matplotlib inline
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from IPython.display import display
from scipy.stats import norm, pareto, uniform, weibull_min

# Use a zero relative MIP gap to report a proven optimum, within solver tolerances.
lp_solver.options['mip_rel_gap'] = 0.0

## Exercise 1: the farmer's problem and its variants

A farmer can plant up to 500 acres of wheat, corn and sugar beets before
the weather is known. The three weather scenarios are equally likely.
Purchases and sales can respond to the weather.

| Crop | Planting cost (€/acre) | Feed needed (tons) | Sale price (€/ton) | Purchase price (€/ton) |
|:---|---:|---:|---:|---:|
| Wheat | 150 | 200 | 170 | 238 |
| Corn | 230 | 240 | 150 | 210 |
| Sugar beets | 260 | 0 | 36 up to 6,000 tons, 10 above | Not purchased |

| Weather | Wheat yield | Corn yield | Beet yield |
|:---|---:|---:|---:|
| Good (`H`) | 3 | 3.6 | 24 |
| Average (`M`) | 2.5 | 3 | 20 |
| Bad (`L`) | 2 | 2.4 | 16 |

Yields are in tons per acre. The farmer can buy wheat or corn to cover a
feed shortfall and can leave some of the harvest unused.
The [MO-book farmer notebook](https://mobook.github.io/MO-book/notebooks/10/04-farmer-problem.html)
uses the same base data.

**(a)** Complete the scenario balance and beet-quota rows in `farmer_model`.
Solve the extensive form and report the planting, scenario profits and
expected profit. Identify the first-stage and recourse decisions. Why must
planting be shared across scenarios? Give a feasible trade in each scenario
for an arbitrary feasible planting.

In [ ]:
CROPS = ['W', 'C', 'S']
crop_names = {'W': 'Wheat', 'C': 'Corn', 'S': 'Sugar beets'}
planting_cost = {'W': 150.0, 'C': 230.0, 'S': 260.0}
mean_yield = {'W': 2.5, 'C': 3.0, 'S': 20.0}
feed = {'W': 200.0, 'C': 240.0}
sale_price = {'W': 170.0, 'C': 150.0}
purchase_price = {'W': 238.0, 'C': 210.0}
weather = {'H': 1.2, 'M': 1.0, 'L': 0.8}

In [ ]:
def farmer_model(yield_factors=None, price_factors=None, probabilities=None):
    factors = dict(weather if yield_factors is None else yield_factors)
    if not factors or any(not np.isfinite(v) or v < 0 for v in factors.values()):
        raise ValueError('Supply nonnegative yields for at least one scenario.')
    prices = {s: 1.0 for s in factors} if price_factors is None else dict(price_factors)
    weights = {s: 1.0 for s in factors} if probabilities is None else dict(probabilities)
    if set(prices) != set(factors) or set(weights) != set(factors):
        raise ValueError('Scenario labels must agree.')
    if any(not np.isfinite(v) or v <= 0 for v in prices.values()):
        raise ValueError('Price factors must be positive.')
    if any(not np.isfinite(v) or v < 0 for v in weights.values()) or sum(weights.values()) <= 0:
        raise ValueError('Scenario weights must be nonnegative with a positive sum.')
    total_weight = sum(weights.values())

    m = pyo.ConcreteModel('Farmer')
    m.CROPS = pyo.Set(initialize=CROPS)
    m.FEED = pyo.Set(initialize=['W', 'C'])
    m.SCENARIOS = pyo.Set(initialize=list(factors))
    m.probability = pyo.Param(m.SCENARIOS, initialize={s: weights[s] / total_weight for s in factors})
    m.plant = pyo.Var(m.CROPS, domain=pyo.NonNegativeReals, bounds=(0, 500))
    m.land = pyo.Constraint(expr=sum(m.plant[c] for c in m.CROPS) <= 500)
    m.buy = pyo.Var(m.FEED, m.SCENARIOS, domain=pyo.NonNegativeReals)
    m.sell = pyo.Var(m.CROPS, m.SCENARIOS, domain=pyo.NonNegativeReals)
    m.sell_extra = pyo.Var(m.SCENARIOS, domain=pyo.NonNegativeReals)

    # Add the feed balances, beet availability rows and beet sales quota.
    raise NotImplementedError('Complete the scenario constraints in 1(a).')

    m.first_stage_profit = pyo.Expression(expr=-sum(planting_cost[c] * m.plant[c] for c in m.CROPS))

    @m.Expression(m.SCENARIOS)
    def recourse_profit(m, s):
        return (sum(prices[s] * (sale_price[c] * m.sell[c, s] - purchase_price[c] * m.buy[c, s])
                    for c in m.FEED)
                + 36 * m.sell['S', s] + 10 * m.sell_extra[s])

    @m.Expression(m.SCENARIOS)
    def scenario_profit(m, s):
        return m.first_stage_profit + m.recourse_profit[s]

    m.total_expected_profit = pyo.Objective(
        expr=sum(m.probability[s] * m.scenario_profit[s] for s in m.SCENARIOS),
        sense=pyo.maximize,
    )
    return m

In [ ]:
def planting_plan(model):
    return pd.Series({c: pyo.value(model.plant[c]) for c in model.CROPS}, name='Acres')


def farm_report(model):
    display(planting_plan(model).rename(index=crop_names).round(2).to_frame())
    profits = pd.Series({s: pyo.value(model.scenario_profit[s]) for s in model.SCENARIOS},
                        name='Scenario profit (€)')
    display(profits.round(2).to_frame())
    print(f'Expected profit: €{pyo.value(model.total_expected_profit):,.2f}')

In [ ]:
# Complete farmer_model, then solve and report the three-scenario model.
# Save the model as farmer_rp and its objective as farmer_optimum.

**(b)** Wheat and corn prices fall by 10% in good weather and rise by 10% in
bad weather. Apply the changes to both purchase and sale prices. Beet prices
remain fixed. Modify the extensive form, solve it, and compare with (a).

| Weather | Wheat sale | Corn sale | Wheat purchase | Corn purchase |
|:---|---:|---:|---:|---:|
| Good | 153 | 135 | 214.2 | 189 |
| Average | 170 | 150 | 238 | 210 |
| Bad | 187 | 165 | 261.8 | 231 |

Prices are in euros per ton. The `price_factors` argument applies only to
wheat and corn.

In [ ]:
# Use the same constructor with scenario-dependent wheat and corn price factors.

**(c)** Return to the fixed prices in (a). The farm has four fields of 185,
145, 105 and 65 acres. Each field can be left idle or planted with a single
crop. Add binary field-assignment variables to a model returned by
`farmer_model` and link them to the crop acreages. Solve and report the
field assignments, crop acreages and expected profit.

In [ ]:
# Add binary assignment variables and one at-most-one-crop row per field.
# Link each crop's acreage to the sizes of its assigned fields.
# Use a threshold of 0.5 when displaying binary selections.

**(d)** Find the planting that maximizes worst-case profit. For any fixed
planting, higher yields cannot reduce optimal trading profit at the prices
in (a), so the worst case occurs in bad weather.

Solve the bad-weather LP. Fix its three acreages at full precision in the
three-scenario model and reoptimize purchases and sales. How much expected
profit does this planting give up relative to (a)?

In [ ]:
# Solve farmer_model(yield_factors={'L': 0.8}).
# Fix its complete planting vector in a fresh three-scenario model and solve again.
# Compare its expected profit with farmer_optimum from (a).

**(e)** Use the prices from (a) and maximize expected profit subject to a
profit of at least €58,000 in every scenario. How much expected profit
does this cost relative to (a)?

Repeat for floors of €48,000, €50,000, €52,000, €54,000, €56,000 and €56,800,
and for the optimal bad-weather profit from (d). Plot the expected-profit
loss against the floor, including the €58,000 case, and compare the crop
acreages. The MO-book farmer notebook also uses the €56,800 floor.

In [ ]:
# Add a lower bound on scenario_profit[s] for every scenario.
# Include the maximin floor from (d), and tabulate profits, losses and all three acreages.
# Plot both the loss and the planting changes.

**(f)** Turn the calculation in (d) into a function,
`evaluate_planting(plan, yield_factors=None, probabilities=None)`. Fix all
three acreages, reoptimize purchases and sales, and return the net profit
in each scenario and its probability-weighted mean. Include planting costs
in each scenario profit.

Use the expected profit from (a) for RP. Solve the mean-yield problem for
the expected-value (EV) planting and evaluate it on the original three
scenarios to obtain EEV. For the wait-and-see value (WS), optimize planting
and trading separately in each scenario and average the resulting profits.
Order RP, EEV and WS, then compute VSS and EVPI and check their signs.

Use L8's profit convention. For the cost convention, see Sheet 4 Exercise
3(d). Check your function on the planting from (d).

In [ ]:
def evaluate_planting(plan, yield_factors=None, probabilities=None):
    # Build a fresh model, fix every planting variable, and reoptimize recourse.
    # Return scenario_profits and expected_profit, keeping the full planting precision.
    raise NotImplementedError('Refactor the evaluation from 1(d).')

# Compute EV, EEV, WS, VSS and EVPI, then reproduce the evaluation in (d).

**(g) Tail risk of a fixed planting.** Implement the auxiliary-variable LP
from L8 as `cvar_from_losses(losses, alpha)` for equally likely losses and
$0<\alpha<1$. The threshold $\eta$ is unrestricted in sign. Check the LP
against a hand calculation for the losses $(1,2,3,4)$ at $\alpha=0.6$.

Use `evaluate_planting` from (f) on the RP planting from (a) and L8's
planting of $(100,100,300)$ acres of wheat, corn and beets. Take each loss
to be the negative of net profit. For each planting, report the scenario
profits, expected profit and $\operatorname{CVaR}_{1/2}$ of the loss.
Which planting does each criterion favor? By how much does the average
profit over the worst half of the probability mass change?

In [ ]:
# Define cvar_from_losses(losses, alpha) using L8's auxiliary variables.
# Compare it with a hand calculation for the losses (1, 2, 3, 4) at alpha = 0.6.
# Evaluate both fixed plantings, then compare expected profit and loss CVaR.

## Exercise 2: a newsvendor model for seafood inventory

A seafood distribution center buys tuna before observing retailer demand.
The purchase price is $c=10$, the sale price is $p=25$, and the cost of
storing unsold tuna is $h=5$, in euros per ton. For an order $x\ge0$ tons
and demand $z\ge0$ tons, the profit is

$$
f(x,z)=p\min(x,z)-cx-h(x-z)^+.
$$

Use uniform demand on $[50,150]$, Pareto demand with $x_m=50$ and
$\alpha=2$, and Weibull demand with shape $k=2$ and scale $\lambda=113$.
These are the parameters in L8's SAA example. The
[MO-book seafood notebook](https://mobook.github.io/MO-book/notebooks/09/03-seafood-distribution-center.html)
uses the same model with different parameters.

**(a)** Find each optimal order using L7's quantile formula
$x^*=F^{-1}(r)$, where $r=(p-c)/(p+h)$ is the critical ratio. Use the
distribution objects below or implement the inverse CDFs directly. For the
Pareto law, $F^{-1}(q)=50/(1-q)^{1/2}$. Plot the CDFs and mark the optimal
orders.

In [ ]:
c, p, h = 10.0, 25.0, 5.0
distributions = {
    'Uniform': uniform(loc=50, scale=100),
    'Pareto': pareto(b=2, scale=50),
    'Weibull': weibull_min(c=2, scale=113),
}


def newsvendor_profit(order, demands):
    demands = np.asarray(demands, dtype=float)
    return p * np.minimum(order, demands) - c * order - h * np.maximum(order - demands, 0)

In [ ]:
# Compute the critical ratio and optimal orders, then plot the three CDFs.

**(b)** Solve the deterministic LP for each distribution, setting demand
to its population mean. The uniform and Pareto means are 100. The Weibull
mean is $113\Gamma(3/2)\approx100.1436426$.

Complete the two recourse constraints in `newsvendor_model`. For a single
deterministic demand, pass a one-element array. Use `fixed_order` to fix an
order when you evaluate it in (c).

In [ ]:
def newsvendor_model(sample, fixed_order=None):
    sample = np.asarray(sample, dtype=float)
    if sample.ndim != 1 or not len(sample) or np.any(~np.isfinite(sample)) or np.any(sample < 0):
        raise ValueError('Supply a nonempty vector of nonnegative demands.')
    m = pyo.ConcreteModel('Newsvendor')
    m.SCENARIOS = pyo.RangeSet(0, len(sample) - 1)
    m.x = pyo.Var(domain=pyo.NonNegativeReals)
    if fixed_order is not None:
        if not np.isfinite(fixed_order) or fixed_order < 0:
            raise ValueError('The fixed order must be nonnegative.')
        m.x.fix(float(fixed_order))
    m.y = pyo.Var(m.SCENARIOS, domain=pyo.NonNegativeReals)
    m.z = pyo.Var(m.SCENARIOS, domain=pyo.NonNegativeReals)

    # Bound sales by demand and balance sales plus leftovers with the order.
    raise NotImplementedError('Complete the recourse constraints in 2(b).')

    m.total_expected_profit = pyo.Objective(
        expr=-c * m.x + sum(p * m.y[s] - h * m.z[s] for s in m.SCENARIOS) / len(sample),
        sense=pyo.maximize,
    )
    return m

In [ ]:
# Solve one deterministic model per distribution, using its own population mean.
# Retain these EV orders for (c)–(e).

**(c)** Use the extensive form to evaluate each EV order from (b) on the
3,000 draws for its distribution. Report its mean profit on that sample.
The next cell reproduces the draws for L8's slide *Convergence of SAA: the
practice* using root seed 2026 and a separate stream for each distribution.
Keep the arrays for (d) and (e).

In [ ]:
N_NEWS = 3000
news_seeds = np.random.SeedSequence(2026).spawn(3)
news_rngs = [np.random.default_rng(seed) for seed in news_seeds]
training_samples = {
    'Uniform': news_rngs[0].uniform(50, 150, N_NEWS),
    'Pareto': 50 * (news_rngs[1].pareto(2, N_NEWS) + 1),
    'Weibull': 113 * news_rngs[2].weibull(2, N_NEWS),
}

In [ ]:
# Evaluate the three EV orders on training_samples without changing the orders.
# Save each model and its sample profit for (d).

**(d)** Optimize the order on each sample from (c). Report the order and
how much mean profit it gains over the population-mean EV order on the same
observations. Explain why the gain is nonnegative. Give the two central
order statistics of each sample and locate the optimized order relative
to them.

In [ ]:
# Optimize on the samples from (c) and compare with the fixed EV orders.
# For the even sample size, find the two central order statistics.

**(e)** Reproduce L8's SAA convergence plot. For $n=25,50,\ldots,3000$,
use the first $n$ observations from each retained array. The critical ratio
is $1/2$, so an empirical median gives an optimal order for each prefix.

Plot the orders against $n$ and mark the exact orders from (a) with
horizontal dashed lines. Give the absolute order error for each distribution
at $n=3000$. Do the errors decrease at every step? Relate the trajectories
to the convergence statement in L8.

In [ ]:
sample_sizes = np.arange(25, N_NEWS + 1, 25)
# Use the first n observations of each retained sample.

In [ ]:
# Compute an empirical median for every prefix and distribution.
# Plot the trajectories with the analytical quantiles from (a).
# Report the final absolute errors and explain whether improvement is monotone.

## Exercise 3: airline seat allocation

Use the formulation in **Sheet 4 Exercise 3(a)**. The plane has space for
200 economy seats. A first-class seat takes the space of two economy seats,
and a business-class seat takes the space of 1.5. Ticket revenues are 3, 2
and 1 times the economy fare for first, business and economy, respectively.

The three demand scenarios are equally likely.

| Scenario | First | Business | Economy |
|:---|---:|---:|---:|
| Morning and evening | 20 | 50 | 200 |
| Weekend | 10 | 25 | 175 |
| Midday | 5 | 10 | 150 |

Choose integer seat counts $x_c$ before observing demand. Tickets are sold
within each class. For fixed seats and integer demand $\xi_{c,s}$, positive
ticket revenue makes optimal sales equal to $\min(x_c,\xi_{c,s})$, so the
ticket-sale variables can be continuous.

The [MO-book aircraft notebook](https://mobook.github.io/MO-book/notebooks/10/01-aircraft-seat-allocation.html)
develops the model in stages. For this lab, use the demand table above and
the forecasts specified in (b) and (c).

**(a)** Complete and solve `aircraft_stochastic`. Use one constraint for
total space and maximize the average scenario revenue. Report the
allocation, tickets sold, unused seats, unmet demand and expected revenue.

In [ ]:
CLASSES = ['F', 'B', 'E']
demand = pd.DataFrame([[20, 50, 200], [10, 25, 175], [5, 10, 150]],
    columns=CLASSES, index=['Morning/evening', 'Weekend', 'Midday'])
capacity = 200
revenue_factor = pd.Series({'F': 3.0, 'B': 2.0, 'E': 1.0})
seat_factor = pd.Series({'F': 2.0, 'B': 1.5, 'E': 1.0})


def integer_demand(data):
    values = data.loc[:, CLASSES].to_numpy(dtype=float)
    if (not len(values) or np.any(~np.isfinite(values)) or np.any(values < 0)
            or not np.allclose(values, np.rint(values), rtol=0, atol=1e-8)):
        raise ValueError('The airline model requires nonnegative integer demands.')
    return pd.DataFrame(np.rint(values).astype(int), columns=CLASSES, index=data.index)


def installed_seats(model):
    raw = np.array([pyo.value(model.seats[c]) for c in CLASSES])
    if not np.allclose(raw, np.rint(raw), rtol=0, atol=1e-6):
        raise ValueError('The solution has noninteger seat counts.')
    return pd.Series(np.rint(raw).astype(int), index=CLASSES, name='Installed seats')


def airline_outcomes(seats, data):
    data = integer_demand(data)
    seats = pd.Series(seats).reindex(CLASSES)
    if (seats.isna().any() or (seats < 0).any()
            or not np.allclose(seats, np.rint(seats), rtol=0, atol=1e-8)
            or seats.dot(seat_factor) > capacity + 1e-7):
        raise ValueError('Supply a feasible integer seat allocation.')
    tickets = data.clip(upper=seats, axis=1)
    first_failure = data['F'] > seats['F']
    combined_failure = data['F'] + data['B'] > seats['F'] + seats['B']
    return {'tickets': tickets, 'unused': seats - tickets, 'spillage': data - tickets,
            'revenue': tickets.dot(revenue_factor), 'first_failure': first_failure,
            'combined_failure': combined_failure, 'joint_failure': first_failure | combined_failure}


def seat_report(seats, data):
    outcomes = airline_outcomes(seats, data)
    display(pd.DataFrame({'Installed seats': seats,
                          'Economy-equivalent space': seats * seat_factor}))
    print(f'Total space: {seats.dot(seat_factor):g} / {capacity}')
    if len(data) <= 10:
        for key, label in [('tickets', 'Tickets sold'), ('unused', 'Unused seats'), ('spillage', 'Unmet demand')]:
            print(label)
            display(outcomes[key])
    else:
        display(pd.DataFrame({key: outcomes[key].mean() for key in ['tickets', 'unused', 'spillage']}).round(3))
    print(f'Mean revenue: {outcomes["revenue"].mean():.3f} economy-ticket units')
    return outcomes

In [ ]:
def aircraft_stochastic(data):
    data = integer_demand(data)
    if not data.index.is_unique:
        raise ValueError('Scenario labels must be unique.')
    m = pyo.ConcreteModel('Aircraft seat allocation')
    m.CLASSES = pyo.Set(initialize=CLASSES)
    m.SCENARIOS = pyo.Set(initialize=data.index)
    m.seats = pyo.Var(m.CLASSES, domain=pyo.NonNegativeIntegers)
    m.tickets = pyo.Var(m.CLASSES, m.SCENARIOS, domain=pyo.NonNegativeReals)

    # Add one physical-capacity row and the demand and seat limits for ticket sales.
    raise NotImplementedError('Complete the airline constraints in 3(a).')

    m.revenue = pyo.Objective(
        expr=sum(revenue_factor[c] * m.tickets[c, s] for c in CLASSES for s in m.SCENARIOS) / len(data),
        sense=pyo.maximize,
    )
    return m

In [ ]:
# Solve aircraft_stochastic(demand), then use installed_seats and seat_report.
# Compare the reported mean revenue with the model's averaged objective.

**(b)** Require first-class demand to fit in first-class seats with
probability at least 98%, and combined first- and business-class demand to
fit in their combined seats with probability at least 95%.

Use the independent Gaussian forecasts in **Sheet 4 Exercise 5(a)**,
denoted here by $\eta_F\sim\mathcal N(16,16)$ and
$\eta_B\sim\mathcal N(30,48)$. The second parameter is the variance. The
equivalent capacity bounds are

$$
x_F\ge16+\Phi^{-1}(0.98)\sqrt{16},\qquad
x_F+x_B\ge46+\Phi^{-1}(0.95)\sqrt{64}.
$$

For integer seats these become $x_F\ge25$ and $x_F+x_B\ge60$. Add the bounds
to your model in (a) and keep its three scenarios for calculating revenue.
Explain the changes in allocation and revenue. Ticket sales still use
the own-class limits in (a).

In [ ]:
sheet_mean = np.array([16.0, 30.0])
sheet_covariance = np.diag([16.0, 48.0])

In [ ]:
# Define gaussian_capacity_bounds(mean, covariance, first_level=0.98, combined_level=0.95).
# Include twice the covariance in the combined variance.
# Define aircraft_cc by extending aircraft_stochastic with the two capacity rows.
# Solve with sheet_mean and sheet_covariance and compare with (a).

**(c)** Now use the forecast from L8's *Seat allocation: correlated
nonnegative demand* slide. In first/business/economy order, the latent
vector $\eta$ is Gaussian with mean and covariance

$$
\mu=(12,28,175),\qquad
\Sigma=\begin{pmatrix}3.5&3.7&2.5\\3.7&6.5&7.5\\2.5&7.5&25.2\end{pmatrix}.
$$

L8 clips demand at zero, $z_c=\max(\eta_c,0)$. For ticket counts the lab
also rounds down, $\xi_c=\lfloor z_c\rfloor$. Use the supplied count sample
obtained from the 1,000 latent draws used for the figure.

Derive the 98% bound for $\eta_F$ and the 95% bound for $\eta_F+\eta_B$.
Include covariance in the combined variance, as in Sheet 4 Exercise 5(d).
Impose the resulting bounds on the seats and maximize average ticket revenue
over the integer count scenarios.

In [ ]:
airline_mean = np.array([12.0, 28.0, 175.0])
airline_covariance = np.array([[3.5, 3.7, 2.5], [3.7, 6.5, 7.5], [2.5, 7.5, 25.2]])


def sample_airline(size, seed):
    latent = np.random.default_rng(seed).multivariate_normal(
        airline_mean, airline_covariance, size, method='cholesky')
    counts = np.floor(np.maximum(latent, 0)).astype(int)
    return pd.DataFrame(counts, columns=CLASSES)

In [ ]:
#@title Load the fixed airline count sample
from base64 import b64decode
from zlib import decompress

_lecture_payload = (
    'eNpVWMl220YQnJNNYiNzsESslOz8t/1iSSRAAAQILpKdl0O+LNPVNQPmgoc3GPT0Ul3dPbEpTGsSszVnE5vSjCYy'
    'uelMaN9P9pmbg10pzGT3VOZoArv+j10pzWAW5sG82j0b82KWJrP75Sl7Mvs1tH/1VmZhJcRW/smub+z78u5rZGWe'
    'cVaH01v7rCinwelySmplynsNab19l6+RlXnEv/JMedYR/3Y4683uybBf/grsns58sut72FtDqzOsG2mRSJu8FanZ'
    'wZYWElqsH3DKZFecx3r6TfzTQJr6rbcrT+aK9TNWWmj13b5X2CO6fTZf7PoKO8WTnT1FvSr+X5hHa0UAi0KcEkLb'
    'EBrK+4CvJ7xf7L8au1lb1WRlvtqvMU9RaSk8kyMuotuCp4u3JS49bBzt+hfzA3vUugN8daRFsvMCaQ39nNivDfWP'
    'iCLxpFjXQM4JpwxWq2erj1vJ4X/FRmqjIzG6IMoNYtFD2oWYFJwM9LDsV4xN0LABhgUJGyDKxXpgLOTfEfrsIHOC'
    'zD3/DYiEwnpjAQmJ1fNIJC+gm0peAQMJpEXAnuoWwFI55QDN33higBiteLr4M/HIbBgLkdbirDd44w3rnZdf0caO'
    'T8X8Cl6S9z38eca/NfYPOLezSBNciccm+EeRfEWkrsiUBj6fzB92zwVfJS6FuXmcaJZlNt8l7qJDShvVVzXQeIa0'
    'nWcAydwI6BVMvsIDByJKTv+ALS3+baycDF9FkwAIl7MUJ3v7ryAqZo6L7RodRZTiMKaEjDg5QZ/vwH9P5IgHNGq6'
    '0tHShB5Te5UZtmChDXF7BnLesDLCdpfLM8t1kNkigwZ6VXUTnE/wzAlf/2V2r82f5h2+emWmKw8Lt+xxboNTJsSl'
    'hoY9M10zN4DMCHHXLFa0q3xdCaCnPiPvpQvlOx8+2YgkYFflkAR4EC5S/6uvJti+B6OO5OQE2eQ4QXFbEm/q8xPf'
    '9WuCnUvorzsVnzGR8Mg8+kWrI9gSWSTvIeHIPNJclv019k/Eqq6E9KRgb23lD/R/gMol+HlnLRDPq/ckso/0z+jz'
    '/QnssQV3belP5Y0BvDERUWLFDZpcEdMWcW/gmRNZVHj+jZkVg2kDXz2V56/wVY3MfSGKIjJVxppSs0oKtn/52l1B'
    'h2ecJVX7s7WoBWZurHRLa91fZP6YMU3Bfo/WSxE5XGvNMyQr2zwBUcrDFXE+MIIJ7JVYvDLWAat8ia9bVqsR9vZ3'
    '1a3xdVw9mbLKCFYfYG/JDHoh6iLU5Zg1Ua12cgZapJym0dTK4qxwyFQ0fsCuG5jnDM9oR3TA1519l+qZQLJqIvq/'
    'WeRL7V7DzyEyImJW5ug9SjKnRvPGSDns5cisAtpqVcrgf7UrA2Ye0IHk+HcDG3PGvfdWF8jWErUjBzILyjl4yRV8'
    'myE7MjLDlewtci7I9HdWwJAdV8EKVePfA6tGAhbVDiSGpeKBM/a7rkD2/Mae1tf9En6uzN9Wzgb5m7ED7LnH8fwj'
    'tJKaskK8Aq/nAfwzop/5TW2XqKoB6vKcOztU6j37ooA2Ct4kL5QzG2jeIr43ZrHW2dl7E6uJYjiEP5UNQtji0JXz'
    '9JM/t4L+FXqknD3kGesjNRcUtciRM2zXPuRGZouJ/wKRfcbXyvpQmW0F5hcvffcoKu46tML3AzNP9mChM3TuoZV2'
    '79pvjIja6NFVsNY3wGENtlRu3MNvPfQR31bMOO3BjtB57jNzWnok/iNUHNfzpPTMCRGZ2HcFsC62p3wAq3v21Stf'
    'SVvgpCWb6ewg+tyY10sw/xpfE/YbJTi5YAYdPAMUjOwHZO6g+Y5Y0noqEnT66NAP/CYbLxDxiL1KaXlSMvEFmVhz'
    '0pGdWl+0vu/Ija6/Sjm/tMBVDc8M7LEl7iP945jHMaTrfzSLF/SJoHqFDI2Qm1rH16g7CTAZ+uxrfZ+TQU7BHO/v'
    'OOpImeKxn/jagC2FD3U+StlLN8yLOY9OxJ6sD0R4wJqVsu9t4bGOXZOs//Tdvva6W8jZIMobdJ6qZwUfVr4/DKGJ'
    'VtWAqE7h7YLdSEs2Xptv6CiesJ6xi65ZOwI/8+6QWQ07ap0TA/pNZqKQeHAs98L+Z57Ojqhxr6x9IXJnxXNzP3er'
    'VitEJyA2St+lB8yRnB3gFadcoY9k4pbR7DhxxOy+BMNu+lYsfYWXHH5GxjpBR7Gk/7V7qdiTTPh3IGsFnObmuSPG'
    'SkCGSVHZyzsvbYDAkhy4B//0QOxIfH7yXdBEtgzRU2mfHDG+W3J4d9dbnqCzzmU67fb+hkFyX7hC97vpzO3Jmdc/'
    'WZe1dkScvHL2Qh08cOV9QoQOU07ssTLXxG+oRE+M2js5xNXKijn+g7U4Ag9EnpN37By015pnpR46n/1dikPvkTOj'
    'zoYRe7+vthLpFLMGZyobJ8RwyVl1ZK6FrKcVu9kb71Vcl1XQYyfeRaxQ3xPe0uhUrrZnZM4ruT3g5FKQf05+Vs2I'
    'KO2yRrJK5Hn4glMmb6/efjywC+3YCc84P0HOOzzcs/LObFbja88uIvBss/f1tPIdnbspcvczc7fpbjBadOCtz8Qn'
    'TrIX8rmy2RKY1+l+yTqSUtqFk2zAiT7lBDcQvTHrdU4Nj/DGBVFz3ezKx/QIf86TwsZ28jHql97RzTdRA9lGewmn'
    'T+H71Ri5lngdOtYpN0drpjg+ObL/dEgu6G2dcy/USuvLwueau1dZoF67zrwg5++YC4lnqpFsMN9rudlhjbom2eRm'
    'TPVSyKzJyC3KYwdma3R3Z7IAugJy0aOP+8wqPWcxt17czQI6uVS8M5nItFqRl5hVl5jQtbsIcfMTMSMyVna9aTz7'
    'ie+R3Fv7+UX6+SVx6Lx0QrUaPAcW7MRaMoDeRC2Yg1rNS8aoZ67pjUHwv5uZClGryOSK+QNtd9ye4f6qgOd1sqvo'
    'nzN7eK22n8HGrpvasg62YLyRORvydOVbd/eomjSoBR3vDyN24O5Ob/A30qqPzPKu1iiqU5sFiZ/pdO6rfdS2qJji'
    'JcmIjh1UzMoitTskh4jnY856BeZ37dxK9Htf/A1SQv0VwxkyN2O3uaP+S5+tBzDz0fwH8eJB2g=='
)
_lecture_bytes = decompress(b64decode(_lecture_payload))
if hashlib.sha256(_lecture_bytes).hexdigest() != '1d67fa2e749b9a544cadc7199d6c30445d531876dbb009941ad70f262d1ed765':
    raise RuntimeError('The embedded lecture sample failed its integrity check.')
demand_saa = pd.DataFrame(np.frombuffer(_lecture_bytes, dtype='<u2').reshape(1000, 3).astype(int),
                          columns=CLASSES)
del _lecture_payload, _lecture_bytes

In [ ]:
display(pd.DataFrame({'Latent mean': airline_mean, 'Count sample mean': demand_saa.mean(),
                      'Latent standard deviation': np.sqrt(np.diag(airline_covariance)),
                      'Count sample standard deviation': demand_saa.std()}, index=CLASSES).round(3))
fig, axes = plt.subplots(1, 3, figsize=(10, 3), layout='constrained')
for ax, cls in zip(axes, CLASSES):
    ax.hist(demand_saa[cls], bins=np.arange(demand_saa[cls].min() - 0.5, demand_saa[cls].max() + 1.5))
    ax.set(title=f'Class {cls}', xlabel='Integer demand', ylabel='Scenarios')
plt.show()

In [ ]:
# Recompute the bounds using airline_mean and airline_covariance.
# Solve on demand_saa and report mean revenue for the count scenarios.

## Exercise 4: a sampled joint capacity target

Use the first $N=500$ count scenarios from 3(c), with weight $1/N$ each.
Let $\widehat{\mathbb P}_N$ denote probability under that sample. Require
both capacity inequalities to hold in at least 95% of the scenarios.

$$
\widehat{\mathbb P}_N(\xi_F\le x_F,\ \xi_F+\xi_B\le x_F+x_B)\ge0.95.
$$

The [book's aircraft Model 6](https://mobook.github.io/MO-book/notebooks/10/01-aircraft-seat-allocation.html#model-6-tackling-chance-constraints-using-saa-in-the-case-of-correlated-demand)
uses separate budgets for its individual targets. For a joint target, use
one indicator per scenario, as in L7 and the book's
[economic dispatch model](https://mobook.github.io/MO-book/notebooks/09/04-economic-dispatch.html).

**(a)** Formulate the joint constraint with a binary variable $u_s$ for
each scenario, where $u_s=1$ permits a violation of either capacity row.
Derive the smallest valid big-$M$ constant for each row and scenario over
the original seat domain. Give the integer violation budget for
$\varepsilon=0.05$ and $N=500$.

**(b)** Add the joint constraint to `aircraft_stochastic` without the Gaussian
bounds. Report the optimal allocation, average revenue, number of
permissions and actual first-class, combined and joint failure counts.
Keep the ticket-sale constraints and the revenue contribution for every
scenario, including those with $u_s=1$.

**(c)** Use the same 500 scenarios for all four models. Solve the
unrestricted model, the model with the latent Gaussian bounds from 3(c)
and the supplied `aircraft_individual` model. It enforces individual 98%
and 95% targets on the counts, with budgets of 10 and 25 scenarios.
Include the joint model from (b).

Report average revenue and coverage for the first-class, combined and joint
events under each allocation. How much of the revenue difference between
the latent Gaussian and sampled joint models is already present under the
sampled individual targets? If the last two models return the same optimal
allocation, does that establish equality of their feasible sets? Relate your
answer to Sheet 4 Exercise 4(d).

In [ ]:
joint_training = demand_saa.iloc[:500].copy()
epsilon = 0.05

In [ ]:
def aircraft_individual(data, first_level=0.98, combined_level=0.95):
    if not (0 <= first_level <= 1 and 0 <= combined_level <= 1):
        raise ValueError('Capacity probabilities must lie in [0, 1].')
    data = integer_demand(data)
    m = aircraft_stochastic(data)
    m.first_permission = pyo.Var(m.SCENARIOS, domain=pyo.Binary)
    m.combined_permission = pyo.Var(m.SCENARIOS, domain=pyo.Binary)
    # A success target q requires at least ceil(q*N) successful scenarios.
    first_budget = len(data) - math.ceil(first_level * len(data))
    combined_budget = len(data) - math.ceil(combined_level * len(data))
    m.first_budget = pyo.Param(initialize=first_budget)
    m.combined_budget = pyo.Param(initialize=combined_budget)

    @m.Constraint(m.SCENARIOS)
    def individual_first(m, s):
        first = int(data.loc[s, 'F'])
        return m.seats['F'] + first * m.first_permission[s] >= first

    @m.Constraint(m.SCENARIOS)
    def individual_combined(m, s):
        combined = int(data.loc[s, 'F'] + data.loc[s, 'B'])
        return m.seats['F'] + m.seats['B'] + combined * m.combined_permission[s] >= combined

    m.first_permission_limit = pyo.Constraint(
        expr=sum(m.first_permission[s] for s in m.SCENARIOS) <= m.first_budget)
    m.combined_permission_limit = pyo.Constraint(
        expr=sum(m.combined_permission[s] for s in m.SCENARIOS) <= m.combined_budget)
    return m

In [ ]:
def aircraft_joint(data, epsilon=0.05):
    if not 0 <= epsilon <= 1:
        raise ValueError('epsilon must lie in [0, 1].')
    data = integer_demand(data)
    m = aircraft_stochastic(data)
    m.u = pyo.Var(m.SCENARIOS, domain=pyo.Binary)
    m.violation_budget = pyo.Param(initialize=math.floor(epsilon * len(data)))

    # Use each u[s] in both capacity rows, and add the shared budget.
    raise NotImplementedError('Complete the joint chance constraints in 4(b).')
    return m

In [ ]:
# Solve the joint model. Count actual failures from the demand and seat values,
# independently of the permission variables; report both counts.

In [ ]:
# Solve the unrestricted, Gaussian-formula, supplied individual and joint models on joint_training.
# On those same scenarios, report mean revenue and first, combined and joint coverage.
# Compare the capacity targets and explain whether the feasible sets agree.

For the connection between chance constraints and VaR, see Sheet 4 Exercise
8(c), which asks you to write a probability requirement on profit as a VaR
bound.